<a href="https://colab.research.google.com/github/bisu617/ai-misinformation-detection/blob/main/model-stage1/Bert_Uncased.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Install libraries
# ============================================================
!pip install -q datasets transformers accelerate evaluate scikit-learn

In [ ]:
# ============================================================
# CELL 2: Check GPU
# ============================================================
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ============================================================
# CELL 3: Load Dataset from HuggingFace
# ============================================================
from datasets import load_dataset

dataset = load_dataset("artem9k/ai-text-detection-pile")
print(dataset)

In [ ]:
# ============================================================
# CELL 4: Save to Disk (so you don't re-download next time)
# ============================================================
dataset.save_to_disk("ai_text_detection_pile")


# ============================================================
# CELL 5: Sample 100K train + 10K test
# ============================================================
from datasets import load_from_disk
import re

ds_full = load_from_disk("ai_text_detection_pile")["train"].shuffle(seed=123)

train_part = ds_full.select(range(100_000))
val_part   = ds_full.select(range(100_000, 110_000))
test_part  = ds_full.select(range(110_000, 120_000))

print('Train:', len(train_part))
print('Val:  ', len(val_part))
print('Test: ', len(test_part))

In [ ]:
# ============================================================
# CELL 6: Clean Text
# ============================================================
def clean_text(ex):
    t = ex["text"]
    if t is None:
        t = ""
    t = str(t)
    ex["text"] = re.sub(r"\s+", " ", t).strip()
    return ex

train_part = train_part.map(clean_text)
val_part   = val_part.map(clean_text)
test_part  = test_part.map(clean_text)

train_part = train_part.filter(lambda x: len(x["text"]) >= 30)
val_part   = val_part.filter(lambda x: len(x["text"]) >= 30)
test_part  = test_part.filter(lambda x: len(x["text"]) >= 30)

print('After cleaning:')
print('Train:', len(train_part))
print('Val:  ', len(val_part))
print('Test: ', len(test_part))

In [ ]:
# ============================================================
# CELL 7: Label Encoding
# ============================================================
label2id = {"human": 0, "ai": 1}
id2label = {0: "human", 1: "ai"}

def encode_label(ex):
    return {"label": label2id[ex["source"]]}

train_part = train_part.map(encode_label)
val_part   = val_part.map(encode_label)
test_part  = test_part.map(encode_label)

# Keep only text and label columns
cols_to_keep = ["text", "label"]
train_part = train_part.remove_columns([c for c in train_part.column_names if c not in cols_to_keep])
val_part   = val_part.remove_columns([c for c in val_part.column_names if c not in cols_to_keep])
test_part  = test_part.remove_columns([c for c in test_part.column_names if c not in cols_to_keep])

print('Labels encoded!')
print(train_part)

In [ ]:
# ============================================================
# CELL 8: Tokenize
# ============================================================
from transformers import AutoTokenizer

model_name = 'bert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128, padding='max_length')

train_tok = train_part.map(tokenize, batched=True, remove_columns=['text'])
val_tok   = val_part.map(tokenize,   batched=True, remove_columns=['text'])
test_tok  = test_part.map(tokenize,  batched=True, remove_columns=['text'])

train_tok.set_format('torch')
val_tok.set_format('torch')
test_tok.set_format('torch')

print('Tokenization done!')

In [ ]:
# ============================================================
# CELL 9: Load Model + Define Metrics
# ============================================================
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

acc_metric = evaluate.load('accuracy')
f1_metric  = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1':       f1_metric.compute(predictions=preds,  references=labels, average='binary')['f1']
    }

print('Model loaded!')

In [ ]:
# ============================================================
# CELL 10: Training Arguments + Trainer
# ============================================================
args = TrainingArguments(
    output_dir='roberta_stage1',
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none'
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
# ============================================================
# CELL 11: Train
# Watch Train Loss vs Val Loss in the table
# val_loss going UP while train_loss goes DOWN = overfitting
# ============================================================
train_result = trainer.train()
print('\nTraining complete!')
print(f"Total train time : {train_result.metrics['train_runtime']:.1f}s")
print(f"Train samples/sec: {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# ============================================================
# CELL 12: Overfitting Plot
# ============================================================
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps  = [x['step'] for x in logs if 'loss' in x and 'eval_loss' not in x]
train_losses = [x['loss'] for x in logs if 'loss' in x and 'eval_loss' not in x]
val_steps    = [x['step'] for x in logs if 'eval_loss' in x]
val_losses   = [x['eval_loss'] for x in logs if 'eval_loss' in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label='Train Loss', color='blue')
plt.plot(val_steps,   val_losses,   label='Val Loss',   color='orange', marker='o')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('RoBERTa - Train Loss vs Val Loss (Overfitting Check)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
print('Overfitting plot done!')


In [ ]:
# ============================================================
# CELL 13: Final Test Evaluation (F1 + Accuracy)
# ============================================================
test_results = trainer.evaluate(test_tok)
print('\n========== FINAL TEST RESULTS ==========')
print(f"Accuracy : {test_results['eval_accuracy']:.4f}")
print(f"F1 Score : {test_results['eval_f1']:.4f}")
print(f"Eval Loss: {test_results['eval_loss']:.4f}")
print('=========================================')


In [ ]:
# ============================================================
# CELL 14: Confusion Matrix + Classification Report
# ============================================================
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

preds_output = trainer.predict(test_tok)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print('\n=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Human', 'AI'])
disp.plot(cmap='Blues')
plt.title('RoBERTa - Confusion Matrix (Stage 1)')
plt.tight_layout()
plt.show()
print('Confusion matrix done!')

In [ ]:
# ============================================================
# CELL 15: Save Model
# ============================================================
trainer.save_model("roberta_stage1_model")
tokenizer.save_pretrained("roberta_stage1_model")
print('Model saved!')